# Workforce Scheduling Agent — quick start

BTE 422, Group 5. This notebook installs the environment, checks it with Gymnasium's `check_env`, runs a random week, scores the rule-based benchmarks on fixed validation weeks, and draws one week's schedule. Runtime: about a minute on a free Colab CPU.

In [ ]:
import importlib, subprocess, sys
if importlib.util.find_spec('wsa') is None:   # running on Colab
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/hashimmsk/workforce-scheduling-agent'], check=True)
import gymnasium as gym, numpy as np, wsa
print('gymnasium', gym.__version__, '| wsa', wsa.__version__)

gymnasium 1.4.0 | wsa 0.1.0


## 1. The environment passes Gymnasium's built-in checks

In [ ]:
from gymnasium.utils.env_checker import check_env
env = wsa.make_env()
check_env(env, skip_render_check=True)
print('observation space:', env.observation_space)
print('action space     :', env.action_space)
print(env.company.describe())

observation space: Box(0.0, 2.0, (260,), float32)
action space     : Discrete(51)
Company seed 0: 50 employees (26 full-time, 24 part-time), wages $15.00-$22.00 (mean $18.97), availability 64% of slots


## 2. One random week, slot by slot

In [ ]:
env = wsa.make_env(render_mode='ansi')
obs, info = env.reset(seed=0)
rng = np.random.default_rng(0)
total, done = 0.0, False
while not done:
    action = rng.choice(np.flatnonzero(info['action_mask']))   # uniform over eligible actions
    obs, reward, terminated, truncated, info = env.step(action)
    total += reward
    done = terminated or truncated
print(env.render())
print('return =', round(total), '| summary:', env.summary())

[week complete] payroll $19,960 / budget $22,268 | unfilled 7 | overtime 0 h | eligible now 0
return = -21360 | summary: {'payroll': 19960.0, 'unfilled': 7, 'unfilled_cost': 1400.0, 'overtime_hours': 0.0, 'budget': 22268.4, 'budget_overrun_penalty': 0.0, 'cost': 21360.0, 'n_slots': 140, 'invalid_actions': 0, 'week_seed': 1826701614}


## 3. Benchmarks on fixed validation weeks

Every policy faces the same weeks and the same absences.

In [ ]:
from wsa.benchmarks import BENCHMARKS
from wsa.evaluate import evaluate, results_table
from wsa.splits import VALIDATION_SEEDS
seeds = VALIDATION_SEEDS[:30]
results = {name: evaluate(cls(), seeds) for name, cls in BENCHMARKS.items()}
print(results_table(results))

policy           mean cost   ±95% CI     gap vs greedy    worst 5%  unfilled  OT hours  overrun $
-------------------------------------------------------------------------------------------------
greedy              22,397       297              0.0%      23,698      0.03      26.1          0
manager_rule        22,980       255              2.6%      24,074      0.00       0.0          9
fixed_plan          23,017       293              2.8%      24,190     13.87      10.9          0
random              23,684       295              5.7%      24,974      8.20      15.2          0


## 4. What a week looks like

In [ ]:
from wsa.evaluate import run_episode
from wsa.benchmarks import GreedyPolicy
from wsa.render import plot_week_schedule, plot_policy_costs
env = wsa.make_env()
run_episode(env, GreedyPolicy(), week=wsa.generate_week(env.cfg, seeds[0]))
fig = plot_week_schedule(env, title='Greedy benchmark, validation week 0')
fig2 = plot_policy_costs(results)

## 5. Train an agent (optional, slow)

```
!python scripts/train_ppo.py --timesteps 200000 --seed 0 --eval-weeks 30
!python scripts/train_dqn.py --timesteps 200000 --seed 0 --eval-weeks 30
```
Clone the repo first (`!git clone https://github.com/hashimmsk/workforce-scheduling-agent`) so the scripts are on disk.